# Accessing [KEGG](https://www.kegg.jp/) with BioServices

**KEGG** (Kyoto Encyclopedia of Genes and Genomes) is a comprehensive database
resource integrating genomic, chemical and systemic functional information.

This notebook shows how to:

- Search for pathways and genes
- Retrieve pathway information and gene entries
- Parse compound and drug data
- Map genes to pathways

> **KEGG REST API**: https://www.kegg.jp/kegg/rest/keggapi.html


In [3]:
import matplotlib.pyplot as plt
import pandas as pd
from bioservices import KEGG

k = KEGG(verbose=False)


## 1. Setting the organism

KEGG uses three-letter organism codes. `hsa` refers to *Homo sapiens*.
You can list all available organisms with `k.organisms`.


In [5]:
# Set the organism to human
k.organism = "hsa"
print("Current organism:", k.organism)
print("Total organisms available:", len(k.organismIds))


Current organism: hsa
Total organisms available: 11620


## 2. Searching for pathways

Use `lookfor_pathway` to find pathways matching a keyword.


In [6]:
# Search for B-cell related pathways in human
results = k.lookfor_pathway("B cell")
print("Pathways matching 'B cell':")
for r in results:
    print(' ', r)


Pathways matching 'B cell':
  map04662 B cell receptor signaling pathway


In [7]:
# Search for cancer-related pathways
cancer_pathways = k.lookfor_pathway("cancer")
print(f"Found {len(cancer_pathways)} cancer-related pathways:")
for p in cancer_pathways[:5]:
    print(' ', p)


Found 17 cancer-related pathways:
  map05200 Pathways in cancer
  map05202 Transcriptional misregulation in cancer
  map05206 MicroRNAs in cancer
  map05205 Proteoglycans in cancer
  map05230 Central carbon metabolism in cancer


## 3. Retrieving pathway details

Use `get` to retrieve a full KEGG entry.  Pathway identifiers follow the
pattern `path:<organism><id>`, e.g. `path:hsa04662` for
the B-cell receptor signalling pathway in human.


In [8]:
# Retrieve the B-cell receptor signalling pathway
entry = k.get("path:hsa04662")
print(entry[:600])


ENTRY       hsa04662                    Pathway
NAME        B cell receptor signaling pathway - Homo sapiens (human)
DESCRIPTION B cells are an important component of adaptive immunity. They produce and secrete millions of different antibody molecules, each of which recognizes a different (foreign) antigen. The B cell receptor (BCR) is an integral membrane protein complex that is composed of two immunoglobulin (Ig) heavy chains, two Ig light chains and two heterodimers of Ig-alpha and Ig-beta. After BCR ligation by antigen, three main protein tyrosine kinases (PTKs) -the SRC-family kinase LYN,


In [9]:
# Parse the entry into a dictionary using keggParser
parsed = k.parse(k.get("path:hsa04662"))
print("Pathway name :", parsed.get("NAME"))
print("Pathway class:", parsed.get("CLASS"))
print("Organism     :", parsed.get("ORGANISM"))
genes = parsed.get("GENE", {})
print(f"Number of genes: {len(genes)}")
# Show first 5 genes
for gid, ginfo in list(genes.items())[:5]:
    print(f"  {gid}: {ginfo}")


Pathway name : ['B cell receptor signaling pathway - Homo sapiens (human)']
Pathway class: Organismal Systems; Immune system
Organism     : Homo sapiens (human) [GN:hsa]
Number of genes: 91
  10000: AKT3; AKT serine/threonine kinase 3 [KO:K04456] [EC:2.7.11.1]
  102723407: IGH; immunoglobulin heavy variable 4-38-2-like [KO:K06856]
  102725035: [KO:K06512]
  10288: LILRB2; leukocyte immunoglobulin like receptor B2 [KO:K06512]
  10451: VAV3; vav guanine nucleotide exchange factor 3 [KO:K05730]


## 4. Looking up genes

KEGG gene identifiers take the form `<organism>:<gene>`, e.g. `hsa:673` for
*BRAF* in human.  Use `find` to search by gene name.


In [10]:
# Find the KEGG entry for BRAF in human
braf = k.find("hsa", "BRAF")
print(braf)


hsa:673	BRAF, B-RAF1, B-raf, BRAF-1, BRAF1, NS7, RAFB1; serine/threonine-protein kinase B-raf isoform 1
hsa:100885775	BANCR, LINC00586; BRAF-activated non-protein coding RNA
hsa:10362	HMG20B, BRAF25, BRAF35, HMGX2, HMGXB2, PP7706, SMARCE1r, SOXL, pp8857; SWI/SNF-related matrix-associated actin-dependent regulator of chromatin subfamily E member 1-related



In [11]:
# Retrieve the full gene entry
gene_entry = k.get("hsa:673")  # BRAF
print(gene_entry[:500])


ENTRY       673               CDS       T01001
SYMBOL      BRAF, B-RAF1, B-raf, BRAF-1, BRAF1, NS7, RAFB1
NAME        (RefSeq) serine/threonine-protein kinase B-raf isoform 1
ORTHOLOGY   K04365  B-Raf proto-oncogene serine/threonine-protein kinase [EC:2.7.11.1]
ORGANISM    hsa  Homo sapiens (human)
PATHWAY     hsa01521  EGFR tyrosine kinase inhibitor resistance
            hsa01522  Endocrine resistance
            hsa04010  MAPK signaling pathway
            hsa04012  ErbB signaling pathway
   


In [12]:
# List pathways that contain BRAF (hsa:673)
pathways = k.get_pathway_by_gene("673", "hsa")
print("Pathways containing BRAF:")
for pid, pname in list(pathways.items())[:8]:
    print(f"  {pid}: {pname}")


Pathways containing BRAF:
  hsa01521: EGFR tyrosine kinase inhibitor resistance
  hsa01522: Endocrine resistance
  hsa04010: MAPK signaling pathway
  hsa04012: ErbB signaling pathway
  hsa04015: Rap1 signaling pathway
  hsa04024: cAMP signaling pathway
  hsa04062: Chemokine signaling pathway
  hsa04068: FoxO signaling pathway


## 5. Working with compounds

KEGG COMPOUND (`cpd`) covers small molecules.  Let us fetch information
about glucose.


In [13]:
# Glucose is C00031 in KEGG COMPOUND
glucose = k.get("cpd:C00031")
print(glucose[:600])


ENTRY       C00031                      Compound
NAME        D-Glucose;
            Grape sugar;
            Dextrose;
            Glucose;
            D-Glucopyranose
FORMULA     C6H12O6
EXACT_MASS  180.0634
MOL_WEIGHT  180.16
REMARK      Same as: D00009
COMMENT     alpha-D-Glucose [CPD:C00267]
            beta-D-Glucose [CPD:C00221]
REACTION    R00010 R00015 R00028 R00049 R00063 R00299 R00300 R00301 
            R00302 R00303 R00304 R00305 R00306 R00308 R00327 R00337 
            R00503 R00534 R00550 R00725 R00801 R00837 R00838 R00839 
            R00850 R00874 R00952 R00953 R00960 R01006 R0


In [14]:
# Search for compounds related to ATP
atp_results = k.find("compound", "ATP")
print(atp_results[:400])


cpd:C00002	ATP; Adenosine 5'-triphosphate
cpd:C00131	dATP; 2'-Deoxyadenosine 5'-triphosphate; Deoxyadenosine 5'-triphosphate; Deoxyadenosine triphosphate
cpd:C02739	1-(5-Phospho-D-ribosyl)-ATP; Phosphoribosyl-ATP; N1-(5-Phospho-D-ribosyl)-ATP; 1-(5-Phosphoribosyl)-ATP; 1-(5-Phospho-beta-D-ribosyl)-ATP
cpd:C04727	[Propionyl-CoA:carbon-dioxide ligase (ADP-forming)]; [Propanoyl-CoA:carbon-dioxide lig


## 6. Visualising a pathway

Call `show_pathway` to open the pathway map image in a web browser.
Alternatively, you can retrieve the image URL to embed it.


In [15]:
# Build the URL to the KEGG pathway image (does not open a browser)
pathway_id = "hsa04660"
url = f"https://www.kegg.jp/pathway/{pathway_id}"
print("Pathway URL:", url)

# To open the pathway in a browser, uncomment the next line:
# k.show_pathway(f'path:{pathway_id}')


Pathway URL: https://www.kegg.jp/pathway/hsa04660


---
For more information, please see the
[bioservices.kegg module documentation](https://bioservices.readthedocs.io/en/main/references.html#module-bioservices.kegg)
and the [KEGG REST API documentation](https://www.kegg.jp/kegg/rest/keggapi.html).
